# MAUT
---
_**M**ulti-**A**ttribute **U**tility **T**heory_

In [1]:
import numpy as np
import pandas as pd

## 1. Introduction
---
**MAUT** (_Multi-Attribute Utility Theory_) evaluates **uncertain multiattribute consequences** through utility functions and **expected utility**. Unlike the certainty-based recipe in [MAVT](./mavt.ipynb), preferences are assessed using lotteries and explicitly reflect attitudes toward risk.

This notebook implements an **additive multiattribute utility model**, a particular MAUT form requiring **additive independence**: lotteries with the same marginal distributions for every attribute must be equally preferred. This is a condition on preferences, not statistical independence of outcomes. More general MAUT models allow interactions.

### 1.1. Applications
---
The _MAUT approach_ supports decisions with both tradeoffs and uncertain consequences, such as:

- Service continuity planning
- Event contingency planning
- Technology deployment
- Supply contract selection

An organization compares **3 server support plans** using **annual support cost** (BRL thousand) and **annual service downtime** (hours), both minimized. The support fee excludes the consequences of lost service and is fixed within each plan; downtime depends on whether the year has a major incident.

All data, probabilities, names and preference assessments are **illustrative**. The two scenarios are mutually exclusive and exhaustive within this simplified model.

In [2]:
df = pd.DataFrame([
    ["Managed support", "Routine year", 150, 2],
    ["Managed support", "Major incident", 150, 42],
    ["Premium support", "Routine year", 190, 1],
    ["Premium support", "Major incident", 190, 11],
    ["Self-managed", "Routine year", 120, 0],
    ["Self-managed", "Major incident", 120, 100]
], columns=["alternative", "scenario", "cost", "downtime"])

df = df.set_index(["alternative", "scenario"])

df.style.background_gradient()

## 2. Method steps
---

### 2.1. Scenarios and probabilities
---
Assign a probability $p_s$ to each scenario, with $p_s\geq0$ and $\sum_s p_s=1$. The organization assesses a **0.80** probability of a routine year and **0.20** of a major incident, using the same external event across plans.

Each row describes a **joint consequence** of cost and downtime. Correlated attribute outcomes are allowed; no independent sampling or multiplication of attribute probabilities is needed.

In [3]:
PROBABILITIES = pd.Series({"Routine year": 0.80, "Major incident": 0.20})

assert (PROBABILITIES >= 0).all() and np.isclose(PROBABILITIES.sum(), 1)
assert df.index.is_unique
for alternative, outcomes in df.groupby(level="alternative", sort=False):
    assert set(outcomes.index.get_level_values("scenario")) == set(PROBABILITIES.index)

PROBABILITIES.to_frame("probability")

,probability
Routine year,0.8
Major incident,0.2


### 2.2. Outcome ranges and orientation
---
Fix the utility endpoints before elicitation. **Cost** ranges from BRL **100 to 250 thousand**, and **downtime** from **0 to 100 hours**. For a minimized attribute, orient performance toward its better endpoint:

$$ \large
z_j(x)=\frac{U_j-x}{U_j-L_j}
$$

Here $z=0$ is worst and $z=1$ is best. This coding alone does not determine a utility function or a risk attitude.

In [4]:
LOWER = pd.Series([100, 0], index=df.columns)
UPPER = pd.Series([250, 100], index=df.columns)

assert df.ge(LOWER).all().all() and df.le(UPPER).all().all()
df_z = (UPPER - df)/(UPPER - LOWER)

pd.DataFrame({"worst": UPPER, "best": LOWER})

,worst,best
cost,250,100
downtime,100,0


### 2.3. Elicit single-attribute utilities
---
Holding the other attribute fixed, ask for indifference between a **sure midpoint** and a lottery yielding the best endpoint with probability $p_j$ and the worst otherwise. With endpoint utilities 1 and 0, this assesses $u_j(0.5)=p_j$.

The organization gives **$p_{\mathrm{cost}}=0.50$** and **$p_{\mathrm{downtime}}=0.80$**. Thus 50 certain downtime hours are indifferent to an 80% chance of zero hours and a 20% chance of 100 hours.

Fit the normalized exponential family:

$$ \large
u_j(z)=\begin{cases}
\dfrac{1-e^{-r_jz}}{1-e^{-r_j}}, & r_j\ne0, \\
z, & r_j=0,
\end{cases}
\qquad r_j=2\log\left(\frac{p_j}{1-p_j}\right)
$$

Positive $r_j$ gives a concave increasing utility in desirable performance. The exponential shape is an assumption fitted to the midpoint assessment; further lottery questions should check it in practice.

In [5]:
P_MID = pd.Series([0.50, 0.80], index=df.columns)
RISK_PARAMETERS = 2*np.log(P_MID/(1 - P_MID))

pd.DataFrame({
    "certain midpoint": (LOWER + UPPER)/2,
    "probability of best endpoint": P_MID,
    "r": RISK_PARAMETERS
})

,certain midpoint,probability of best endpoint,r
cost,175.0,0.5,0.000000
downtime,50.0,0.8,2.772589


### 2.4. Evaluate outcome utilities
---
Apply the elicited utility function to **every scenario outcome**. Use the linear limit when the risk parameter is zero. The helper below only evaluates this single-attribute function.

In [6]:
def exponential_utility(z, risk_parameter):
    z = np.asarray(z, dtype=float)
    if np.isclose(risk_parameter, 0):
        return z
    return -np.expm1(-risk_parameter*z)/-np.expm1(-risk_parameter)


df_u = pd.DataFrame(index=df.index)

for col in df.columns:
    r = RISK_PARAMETERS[col]
    df_u[col] = exponential_utility(df_z[col], r)
    assert np.allclose(exponential_utility([0, 0.5, 1], r), [0, P_MID[col], 1])

df_u.style.background_gradient(vmin=0, vmax=1)

### 2.5. Multiattribute scaling constants
---
Normalize overall utility to **0** at the all-worst bundle and **1** at the all-best bundle. Under additive independence:

$$ \large
U(\boldsymbol{x})=\sum_j k_j u_j(x_j), \qquad k_j\geq0,\quad\sum_j k_j=1
$$

Elicit $k_j$ by comparing the sure bundle with attribute $j$ best and all others worst against a lottery between the **overall best** and **overall worst** bundles. The indifference probability of the overall best equals $k_j$.

The assessed probabilities are **0.40** for the cost-only improvement and **0.60** for the downtime-only improvement. They sum to one, as required by this normalized additive model. Do not automatically normalize incompatible lottery assessments; review the preferences or the model form.

In [7]:
SCALING_ASSESSMENTS = pd.DataFrame({
    "cost in sure bundle": [100, 250],
    "downtime in sure bundle": [100, 0],
    "probability of overall best": [0.40, 0.60]
}, index=df.columns)

WEIGHTS = SCALING_ASSESSMENTS["probability of overall best"].copy()
assert (WEIGHTS >= 0).all() and np.isclose(WEIGHTS.sum(), 1)

SCALING_ASSESSMENTS

,cost in sure bundle,downtime in sure bundle,probability of overall best
cost,100,100,0.4
downtime,250,0,0.6


### 2.6. Multiattribute utility in each scenario
---
Multiply each single-attribute utility by its scaling constant and sum the contributions within each joint outcome.

In [8]:
df_contributions = df_u*WEIGHTS
df_contributions["utility"] = df_contributions.sum(axis=1)

df_contributions

cost  downtime   utility
alternative     scenario                                    
Managed support Routine year    0.266667  0.597719  0.864386
                Major incident  0.266667  0.511829  0.778495
Premium support Routine year    0.160000  0.598875  0.758875
                Major incident  0.160000  0.585736  0.745736
Self-managed    Routine year    0.346667  0.600000  0.946667
                Major incident  0.346667  0.000000  0.346667

### 2.7. Expected utility
---
Take the probability-weighted average of the **outcome utilities**:

$$ \large
EU(a)=\sum_s p_sU(\boldsymbol{x}_{as})
$$

Show both the scenario contributions and the expected contribution of each attribute.

In [9]:
p = PROBABILITIES.reindex(df.index.get_level_values("scenario")).to_numpy()

df_scenario = df_contributions[["utility"]].copy()
df_scenario["probability"] = p
df_scenario["weighted utility"] = df_scenario["utility"]*p

df_scenario

utility  probability  weighted utility
alternative     scenario                                               
Managed support Routine year    0.864386          0.8          0.691509
                Major incident  0.778495          0.2          0.155699
Premium support Routine year    0.758875          0.8          0.607100
                Major incident  0.745736          0.2          0.149147
Self-managed    Routine year    0.946667          0.8          0.757333
                Major incident  0.346667          0.2          0.069333

In [10]:
df_expected = (df_u*WEIGHTS).mul(p, axis=0).groupby(level="alternative", sort=False).sum()
df_expected["expected utility"] = df_expected.sum(axis=1)

df_expected

,cost,downtime,expected utility
alternative,,,
Managed support,0.266667,0.580541,0.847208
Premium support,0.160000,0.596248,0.756248
Self-managed,0.346667,0.480000,0.826667


### 2.8. Ranking
---
Rank by **decreasing expected utility**. Utilities represent preferences, rather than probabilities of success.

In [11]:
df_rank = df_expected[["expected utility"]].copy()
df_rank["ranking"] = (
    df_rank["expected utility"].rank(ascending=False, method="min").astype(int)
)

df_rank.sort_values("ranking").style.background_gradient(subset=["expected utility"])

,expected utility,ranking
alternative,,
Managed support,0.847208,1
Self-managed,0.826667,2
Premium support,0.756248,3


**Managed support** is preferred. It limits major-incident downtime at a moderate cost. Premium support further reduces downtime, but its cost is not justified under these preferences. Self-management is cheapest and performs well routinely, but its severe incident outcome reduces expected utility.

### 2.9. Utility of the mean is different
---
Nonlinear utility generally gives **$E[U(X)]\ne U(E[X])$**. The table below compares the valid expected utility with the utility obtained by replacing uncertain outcomes with their means. It also shows a **linear-utility comparison**, representing different risk preferences while keeping the endpoint scaling constants fixed.

In [12]:
df_mean = df.mul(p, axis=0).groupby(level="alternative", sort=False).sum()
df_mean_z = (UPPER - df_mean)/(UPPER - LOWER)
df_mean_u = pd.DataFrame(index=df_mean.index)

for col in df.columns:
    df_mean_u[col] = exponential_utility(df_mean_z[col], RISK_PARAMETERS[col])

df_comparison = pd.DataFrame({
    "expected utility": df_expected["expected utility"],
    "utility at mean outcomes": df_mean_u.mul(WEIGHTS).sum(axis=1),
    "linear-utility comparison": (
        df_z.mul(WEIGHTS).sum(axis=1).mul(p)
        .groupby(level="alternative", sort=False).sum()
    )
})

df_comparison

,expected utility,utility at mean outcomes,linear-utility comparison
alternative,,,
Managed support,0.847208,0.853886,0.806667
Premium support,0.756248,0.756531,0.742000
Self-managed,0.826667,0.917023,0.826667


Using mean outcomes first would incorrectly favor **self-management** under the assessed nonlinear utilities. The linear-utility comparison also favors it, but describes a different preference model. This distinction is why the scenario utilities are computed before taking expectations.

## 3. Examples
---

### 3.1. Community festival venue
---
Compare a **covered site**, an **open-air site**, and an **indoor hall** using **attendance** (maximize) and **total cost** (BRL thousand, minimize). Weather changes both attendance and cost; each row is a joint consequence.

The data, weather probabilities and preferences are **illustrative**. The organizer assumes additive independence, uses endpoint scaling constants **0.70** for attendance and **0.30** for cost, and assesses midpoint lottery probabilities **0.75** and **0.50**, respectively. Attendance ranges from **0 to 1,200**, and cost from **40 to 100 thousand**.

In [13]:
df_ex1 = pd.DataFrame([
    ["Covered site", "Dry", 900, 70],
    ["Covered site", "Wet", 800, 80],
    ["Open-air site", "Dry", 1200, 50],
    ["Open-air site", "Wet", 150, 65],
    ["Indoor hall", "Dry", 750, 85],
    ["Indoor hall", "Wet", 750, 85]
], columns=["alternative", "scenario", "attendance", "cost"])

df_ex1 = df_ex1.set_index(["alternative", "scenario"])

df_ex1.style.background_gradient()

In [14]:
PROBABILITIES_ex1 = pd.Series({"Dry": 0.60, "Wet": 0.40})
LOWER_ex1 = pd.Series([0, 40], index=df_ex1.columns)
UPPER_ex1 = pd.Series([1200, 100], index=df_ex1.columns)
OBJECTIVES_ex1 = pd.Series(["max", "min"], index=df_ex1.columns)
P_MID_ex1 = pd.Series([0.75, 0.50], index=df_ex1.columns)
RISK_PARAMETERS_ex1 = 2*np.log(P_MID_ex1/(1 - P_MID_ex1))
WEIGHTS_ex1 = pd.Series([0.70, 0.30], index=df_ex1.columns)

assert np.isclose(PROBABILITIES_ex1.sum(), 1)
assert np.isclose(WEIGHTS_ex1.sum(), 1)
assert df_ex1.ge(LOWER_ex1).all().all() and df_ex1.le(UPPER_ex1).all().all()

pd.DataFrame({
    "objective": OBJECTIVES_ex1,
    "lower": LOWER_ex1,
    "upper": UPPER_ex1,
    "midpoint lottery probability": P_MID_ex1,
    "r": RISK_PARAMETERS_ex1,
    "scaling constant": WEIGHTS_ex1
})

,objective,lower,upper,midpoint lottery probability,r,scaling constant
attendance,max,0,1200,0.75,2.197225,0.7
cost,min,40,100,0.50,0.000000,0.3


In [15]:
# Oriented performances and single-attribute utilities
df_ex1_z = (df_ex1 - LOWER_ex1)/(UPPER_ex1 - LOWER_ex1)
df_ex1_z["cost"] = 1 - df_ex1_z["cost"]

df_ex1_u = pd.DataFrame(index=df_ex1.index)
for col in df_ex1.columns:
    df_ex1_u[col] = exponential_utility(df_ex1_z[col], RISK_PARAMETERS_ex1[col])

df_ex1_u.style.background_gradient(vmin=0, vmax=1)

In [16]:
# Joint-outcome utilities and their probability-weighted contributions
p_ex1 = PROBABILITIES_ex1.reindex(df_ex1.index.get_level_values("scenario")).to_numpy()
df_ex1_contributions = df_ex1_u*WEIGHTS_ex1
df_ex1_scenario = df_ex1_contributions.copy()
df_ex1_scenario["utility"] = df_ex1_contributions.sum(axis=1)
df_ex1_scenario["probability"] = p_ex1
df_ex1_scenario["weighted utility"] = df_ex1_scenario["utility"]*p_ex1

df_ex1_scenario

attendance   cost   utility  probability  \
alternative   scenario                                             
Covered site  Dry         0.635946  0.150  0.785946          0.6   
              Wet         0.605493  0.100  0.705493          0.4   
Open-air site Dry         0.700000  0.250  0.950000          0.6   
              Wet         0.189129  0.175  0.364129          0.4   
Indoor hall   Dry         0.588043  0.075  0.663043          0.6   
              Wet         0.588043  0.075  0.663043          0.4   

                        weighted utility  
alternative   scenario                    
Covered site  Dry               0.471567  
              Wet               0.282197  
Open-air site Dry               0.570000  
              Wet               0.145652  
Indoor hall   Dry               0.397826  
              Wet               0.265217

In [17]:
df_ex1_expected = (
    df_ex1_contributions.mul(p_ex1, axis=0)
    .groupby(level="alternative", sort=False).sum()
)
df_ex1_expected["expected utility"] = df_ex1_expected.sum(axis=1)
df_ex1_expected["ranking"] = (
    df_ex1_expected["expected utility"].rank(ascending=False, method="min").astype(int)
)

df_ex1_expected.sort_values("ranking").style.background_gradient(subset=["expected utility"])

,attendance,cost,expected utility,ranking
alternative,,,,
Covered site,0.623764,0.130000,0.753764,1
Open-air site,0.495652,0.220000,0.715652,2
Indoor hall,0.588043,0.075000,0.663043,3


The **covered site** leads. It retains most of its attendance under wet weather, while the open-air site's cheaper dry-weather outcome does not offset the severe attendance loss when wet. The indoor hall is reliable but more expensive and attracts fewer attendees. The recommendation follows the assessed probabilities, utility curves and scaling constants.

## 4. References
---

- Keeney, R. L. and Raiffa, H. (1993). [*Decisions with Multiple Objectives: Preferences and Value Trade-Offs*](https://doi.org/10.1017/CBO9781139174084). Chapters 4–6 cover utility assessment and multiattribute preferences under uncertainty; first published in 1976.
- Fishburn, P. C. (1965). [*Independence in Utility Theory with Whole Product Sets*](https://doi.org/10.1287/opre.13.1.28). Establishes an additive representation through preferences over multiattribute gambles.
- Parnell, G. S. (2009). [*Decision Analysis in One Chart*](https://extapps.ksc.nasa.gov/reliability/Documents/Decision_Analysis_in_One_Chart.pdf). Distinguishes value from risk preference and discusses lottery assessment and exponential utility.